# F02-P5 Benefit

**Benefit Quantification, components 5.x.**

Turns the site description produced by the earlier stages into the benefits a project could
claim. Where F02-P2 says what is there and F02-P4 says what may be done with it, this notebook
says what that delivers, first as a qualitative benefit profile, then in carbon terms.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.**

| Component | What it produces | Status |
|---|---|---|
| 5.1 General Benefit | the Triple Win benefits of the activities present, merged | written |
| 5.2 Avoided Emissions from Unplanned Deforestation | avoided loss of standing carbon on the Protect area | written |
| 5.3 ARR Carbon Removal (ex-ante) | ex-ante carbon removal from restoration on the Restore area | written |

5.2 quantifies the Protect side, 5.3 the Restore side. The Manage carbon component is not
started; its qualitative benefits already appear in 5.1.

## Project duration enters here (5.2 and 5.3)

5.2 and 5.3 both depend on a user input other than the AOI polygon: `PROJECT_DURATION_YEARS`,
set in the Setup cell below next to `AOI_PATH`. It is passed as an argument and written into
`values`, so a saved result records the duration it was produced with. 5.1 does not use it.

## Handoff

5.1 and 5.2 have separate run cells and separate upstream stages, so they can be run in any
order and independently. `results` accumulates across the run cells, and the Save cell writes
whatever has been run.

- 5.1 reads `outputs/<aoi_id>__F02-P4-pathway.json` for the activity list from 4.2. Required for 5.1.
- 5.2 reads `outputs/<aoi_id>__F02-P2-general.json` for the deforestation rate from 1.5, and
  `outputs/<aoi_id>__F02-P4-pathway.json` for the QB Avoided Emissions gate. Both required for 5.2.
- 5.3 reads the pathway raster and the AGB raster directly, and uses `PROJECT_DURATION_YEARS`. It
  needs no earlier stage file.

Writes `outputs/<aoi_id>__F02-P5-benefit.json`.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import math
from dataclasses import dataclass

import geopandas as gpd
import numpy as np

from config import *
from common import *

In [2]:
AOI_PATH = r"D:\NBSTOOLV3\AOI1.shp"
aoi_id = "aoi1"                              # must match the F02-P2 and F02-P4 notebooks

# User input. Whole years, the crediting period the user is asking about.
PROJECT_DURATION_YEARS = 20   # <SET: project duration in years>

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)} over {PROJECT_DURATION_YEARS} years")

AOI aoi1: 67,439 ha over 20 years


---
## 5.1 General Benefit

For each pathway present in the AOI, lists the activities that apply and the benefits they bring,
split into the three Triple Win pillars: nature, climate and people.

**Data.** No new layer. Reads the activity list from 4.2 (`F02-P4-pathway.json`), which has joined
the pathway raster to the canonical_v3_activities catalog on `(cat_code, ecosystem)`. Each
activity carries three benefit fields, one per pillar.

**What it does.**

- Groups by pathway (Protect / Manage / Restore). Ineligible and unclassified area carry no
  activity in 4.2, so they contribute nothing here.
- Lists the distinct activities of each pathway, deduplicated by activity id.
- Collects the benefits of those activities, splits each benefit cell on ';', normalizes the text
  (whitespace, trailing period), and deduplicates within each pathway and pillar so each benefit
  is listed once. The catalog vocabulary is closed (17 phrases), so exact matching merges every
  real duplicate.

**No ranking, no area, no render.** This is a plain listing. It does not weight benefits by area,
rank them, or write a narrative sentence. Benefits are sorted alphabetically for a stable order.
The output is the `by_pathway` structure plus two flat tables.

**Output shape.**

```
by_pathway = {
  "Protect": {
    "activities": [{"activity_id": "11", "activity": "Establish protected areas ..."}, ...],
    "benefits": {
      "nature":  ["Enhanced biodiversity and ecosystem functions", ...],
      "climate": ["Increased carbon sequestration and storage", ...],
      "people":  ["Secure land and resource tenure", ...],
    }
  },
  "Manage":  {...},
  "Restore": {...},
}
```

**Downstream use.** The qualitative benefit card of Phase 5, beside 5.2 and the future Manage and
Restore carbon numbers.

In [3]:
# 5.1 General Benefit -------------------------------------------------------------------
# For each pathway present in the AOI: the activities that apply, and the benefits they bring,
# split into the three Triple Win pillars. Benefits are deduplicated so each distinct benefit is
# listed once per pathway.

# Benefit column -> short pillar key, in the order the tool lists them (nature, climate, people).
PILLAR_KEYS = {
    "benefit_nature":  "nature",
    "benefit_climate": "climate",
    "benefit_people":  "people",
}

# The pathways that carry activities, in report order. Ineligible is absent by design: 4.2 gives
# it no activities, so it never reaches this component.
PATHWAY_REPORT_ORDER = ("Protect", "Manage", "Restore")


def _split_benefit_phrases(cell: str) -> list[str]:
    """Split one benefit cell into normalized phrases.

    The catalog stores several benefits per cell, separated by ';'. Normalizing collapses
    whitespace and drops a trailing period, so the same benefit written slightly differently
    merges instead of being listed twice.
    """
    out: list[str] = []
    for part in cell.split(";"):
        phrase = " ".join(part.split()).strip().rstrip(".").strip()
        if phrase:
            out.append(phrase)
    return out


def analyze_general_benefit(pathway_stage: dict) -> ComponentResult:
    """Component 5.1. Per pathway present in the AOI, list its activities and their benefits,
    split into nature, climate and people.

    Reads 4.2's `by_category` output. No new layer, no area weighting, no ranking.
    """
    component = "5.1 General Benefit"

    try:
        by_category = component_values(pathway_stage, "4.2")["by_category"]
    except KeyError:
        return not_applicable(
            component,
            "The activity list (4.2) is not available for this project area. Run F02-P4 for "
            "this AOI first.",
        )
    if not by_category:
        return not_applicable(
            component,
            "The activity list (4.2) found no categories in this project area, so there are no "
            "activities or benefits to list.",
        )

    # pathway -> {"activities": {activity_id: text}, "benefits": {pillar: set(phrases)}}
    collected: dict[str, dict] = {}
    for info in by_category.values():
        activities = info.get("activities", [])
        if not activities:
            continue  # Ineligible / unclassified: no activity, no benefit
        pathway = info.get("pathway", "Unknown")
        slot = collected.setdefault(pathway, {
            "activities": {},
            "benefits": {pillar: set() for pillar in PILLAR_KEYS.values()},
        })
        for activity in activities:
            slot["activities"].setdefault(activity.get("activity_id", ""), activity.get("activity", ""))
            for col, pillar in PILLAR_KEYS.items():
                for phrase in _split_benefit_phrases(activity.get(col, "") or ""):
                    slot["benefits"][pillar].add(phrase)

    if not collected:
        return not_applicable(
            component,
            "No activity in this project area declares a benefit, so no benefits can be listed.",
        )

    by_pathway: dict[str, dict] = {}
    activity_rows: list[dict] = []
    benefit_rows: list[dict] = []
    for pw in PATHWAY_REPORT_ORDER:
        if pw not in collected:
            continue
        # Activities keep the order 4.2 gave them, dominant category first. Benefits are sorted
        # alphabetically for a stable, predictable order.
        activities = [
            {"activity_id": aid, "activity": text}
            for aid, text in collected[pw]["activities"].items()
        ]
        benefits = {
            pillar: sorted(collected[pw]["benefits"][pillar])
            for pillar in PILLAR_KEYS.values()
        }
        by_pathway[pw] = {"activities": activities, "benefits": benefits}

        for row in activities:
            activity_rows.append({"pathway": pw, **row})
        for pillar in PILLAR_KEYS.values():
            for benefit in benefits[pillar]:
                benefit_rows.append({"pathway": pw, "pillar": pillar, "benefit": benefit})

    return ComponentResult(
        component=component,
        applicable=True,
        narrative="",  # no render: by_pathway and the two tables carry everything
        tables={"activities": activity_rows, "benefits": benefit_rows},
        values={
            "by_pathway": by_pathway,
            "pathways_present": list(by_pathway),
        },
        flags=[],
    )

In [4]:
# Run 5.1. It needs only the F02-P4 pathway stage, so it can run on its own, before 5.2.
# `results` accumulates across cells: rerunning a component replaces only its own entry, so 5.1
# and 5.2 can be run in any order and the save cell writes whatever has been run.
try:
    results
except NameError:
    results = {}

pathway = load_results(aoi_id, STAGE_PATHWAY)
results["5.1"] = analyze_general_benefit(pathway)

r = results["5.1"]
print(f"[5.1] {r.component}{'' if r.applicable else '  (not applicable)'}")
if not r.applicable:
    print("     ", r.narrative)
for pw, blk in r.values.get("by_pathway", {}).items():
    print(f"\n{pw}")
    print("  Activities:")
    for a in blk["activities"]:
        print(f"    [{a['activity_id']}] {a['activity']}")
    print("  Benefits:")
    for pillar in ("nature", "climate", "people"):
        items = blk["benefits"][pillar]
        print(f"    {pillar}: " + ("; ".join(items) if items else "-"))
for f in r.flags:
    print("FLAG:", f)

[5.1] 5.1 General Benefit

Protect
  Activities:
    [11] Establish protected areas, corridors, and enforce customary land rights.
  Benefits:
    nature: Enhanced biodiversity and ecosystem functions; Maintenance of ecological connectivity
    climate: Increased carbon sequestration and storage; Reduced emissions from deforestation and degradation
    people: Cultural heritage preservation; Secure land and resource tenure

Manage
  Activities:
    [911] Agroforestry (alley cropping, windbreaks) with zero-tillage, cover crops, and biochar.
    [61] Protect regenerating stands from fire/grazing and apply CBFM.
  Benefits:
    nature: Enhanced biodiversity and ecosystem functions; Improved forest productivity and regeneration; Maintenance of ecological connectivity; Protection of watershed functions
    climate: Increased carbon sequestration and storage; Microclimate regulation
    people: Enhanced food and water security; Secure land and resource tenure; Strengthened social capital and

---
## 5.2 Avoided Emissions from Unplanned Deforestation

**Why "unplanned".** This is the AUD case: deforestation driven by diffuse, unsanctioned pressure,
which is why the baseline comes from a spatial risk model rather than from a document. Avoided
planned deforestation (APD) is a different construct with a different baseline, a land status
overlay showing legally sanctioned conversion, and it is not what this component estimates. The
title says unplanned so the two are not read as interchangeable.

The component is computed on the Protect area, but Protect is the input it runs on, not what it
measures. The output is an avoided emission estimate.

Estimates how much CO2e a Protect project could keep out of the atmosphere over its lifetime, by
projecting the historical deforestation rate forward, placing that projected loss on the highest
risk forest, and reading the carbon standing on it.

**Data.** No new layer. Three layers that other components already declare, read onto one grid:

| Layer | Role | Declared by |
|---|---|---|
| `pathway.tif` band 1 | which pixels are Protect (code 1) | 4.1 |
| `pathway.tif` band 3 | reference ecosystem, for the peat check | 4.1 |
| `prob.tif` | ranks forest pixels by deforestation risk | 1.6 |
| `agb_mgha.tif` | carbon standing on each pixel (belowground derived, see below) | 3.1 |

Plus one number from an earlier stage: `rate_pct` from 1.5.

Belowground biomass is not a separate layer here. There is no mapped BGB raster yet, so it is
derived from aboveground by a fixed root-to-shoot ratio (`ROOT_TO_SHOOT_RATIO`, 0.28): total
biomass is `AGB * (1 + 0.28)`. 5.2 sums the two pools into one number, so unlike the 3.1 pool
split this is not distorted by the ratio being constant.

### The four steps

```
1. protect_ha    = area of pixels that are Protect AND carry a risk value
2. lost_ha(t)    = protect_ha * (1 - exp(-r * t))        r = rate_pct / 100
3. allocate      = take pixels in descending risk order until lost_ha(t) is filled
4. avoided(t)    = sum of AGB * (1 + 0.28) on those pixels * 0.47 * 44/12
```

### Gate: run only when avoided emissions is the right method

Before anything is computed, 5.2 checks the activity catalog through 4.2. Quantification runs only
when the AOI holds at least one Protect-pathway activity whose `QB Avoided Emissions` column is
Yes. That flag is the catalog's own statement that avoided deforestation is the accounting method
for the category, so the gate keeps 5.2 from putting a number on a site the catalog never meant
to be quantified this way. When the gate fails the component returns not applicable with a plain
reason, and no raster is read.

The gate reads 4.2's `by_category` (so 5.2 now depends on the F02-P4 stage as well as F02-P2
general), sums the area of Protect categories that carry a QB Avoided activity, and passes when
that area is above zero. Two things to keep in mind. First, in the current canonical catalog every
Cat 1 (Protect) row is QB Avoided = Yes, so today the gate passes whenever Protect exists with a
matching catalog row; it earns its place by failing closed if the catalog changes or if the
Protect pixels have no activity row at all. Second, the gate is a precondition, not a mask: once it
passes, the quantification still runs on every Protect pixel that carries a risk value, defined
from the raster as before, not only on the categories that tripped the gate. If a future catalog
mixes QB Yes and QB No inside Protect, restricting the pool to the QB Yes categories would be the
next step; it is noted as an open item, not done here.

### Decisions locked

**The rate is borrowed, and it has to be.** Protect is assigned upstream to forest that
persisted from 2014 to 2024. Measuring historical loss inside the Protect area therefore returns
zero by construction, not by observation: the area was selected on the criterion of not having
been deforested. This is a selection effect, so its own history carries no information about its
future and the rate must come from a wider population of forest.

**That wider population is the AOI forest, from 1.5.** The team chose the rate the tool already
has over a precomputed district reference table. Two consequences to keep in view:

1. The AOI average includes degraded and edge forest, which loses faster than intact interior.
   Applying it to Protect leans towards over-estimating the baseline.
2. The rate is measured inside a polygon the user draws. Adding already cleared land to the AOI
   raises `r`, which is then charged to the Protect area. Nothing in the calculation resists
   this.

`values["baseline_rate_source"]` records which population the rate came from, so a later switch
to a district table is visible in saved results rather than silent. `_project_loss_series` takes
the rate as an argument for the same reason.

**Projection is compounding, not linear.** `lost_ha(t) = protect_ha * (1 - exp(-r * t))`. The
Puyravaud rate in 1.5 is an exponential rate, so multiplying a constant ha/year by the duration
would impose a linear shape on an exponential process and over-project long durations. The
compounding form also approaches the Protect area asymptotically instead of crossing it, so the
cap on projected loss is a safeguard rather than something that fires routinely.

**Allocation is by descending risk, and the result of that is conservative.** The highest risk
pixels sit on frontiers and edges, and frontier forest usually carries less biomass than intact
interior. Ranked allocation therefore yields a **lower** figure than spreading the same loss
evenly across the Protect area. That is a property of the method, not a bug, and the difference
is reported in `values["uniform_allocation_tco2e"]` so the two can be compared.

Two limits on the ranking, from the 1.6 markdown cell: `prob.tif` is a relative spatial ranking
and not an absolute probability, and if it is a mosaic of separately fitted regional models the
scale may not be comparable across regions. Both are acceptable here because this component uses
only the **order** of pixels within one AOI, never the level, and never compares one AOI to
another. Risk ties are broken arbitrarily by the sort, which is harmless: tied pixels are
interchangeable by definition of the ranking.

**The annual series can rise, and that is not a bug.** Two quantities move in opposite
directions as the projection runs. The area lost each year falls, because compounding works on a
shrinking stock. The carbon per hectare of that loss rises, because the allocation walks down the
risk ranking and lower risk forest is usually less degraded. Their product has no guaranteed
direction. On a synthetic site with a degraded frontier the annual figure declines gently inside
each stretch of similar forest, then steps up whenever the ranking crosses into denser forest.
Only two things are guaranteed: the annual area lost declines monotonically, and the cumulative
total rises. Nobody should later smooth or "correct" the annual curve into a decline.

**The pixel at the boundary is split, not rounded.** The projected area rarely lands on a whole
pixel. The last pixel contributes the fraction of its area that is needed. Rounding to whole
pixels instead would make the annual series step rather than curve on small sites.

**Emission factor is 100 percent of AGB plus BGB, labelled as an upper bound.** No assumption is
made about what replaces the forest. IPCC and VCS practice takes the difference between the
forest stock and the stock of the land use that follows, which usually leaves 5 to 15 percent
standing. Taking the full stock avoids a parameter that is not calibrated, at the cost of
sitting at the top of the plausible range. The narrative says so.

**No deductions.** Leakage, uncertainty and the non-permanence buffer are all absent. A VCS
buffer alone is commonly 10 to 25 percent. The figure is gross and is **not a creditable
volume**. The team decided this belongs in the documentation only, so the component carries no
`deductions_applied` marker in `values` and the narrative makes no claim about it either. Anyone
reading `total_tco2e` out of the result JSON has to know from here that it is gross. Read the
next section before quoting the number anywhere.

**The narrative names the ecosystem, and it lists every one present.** The word comes from
pathway band 3 restricted to the Protect pool, ordered by area, joined with `oxford_join`, so a
mixed site reads "this forest and peatland ecosystem" rather than being given a single label.
This follows the rule the rest of the tool uses: the AOI is heterogeneous and is never reduced
to one class. Only three words can appear, because `prob.tif` is forest masked and Protect pixels
on grassland, savanna or water carry no risk value and never reach the pool. If one does reach
it, the risk layer and band 3 disagree about what forest is, and the component flags it rather
than inventing a fourth word.

**Grid alignment is required here, unlike in 3.1.** Component 3.1 integrates AGB and BGB
separately on purpose, so the two rasters never have to share a grid. 5.2 cannot do that: it
selects pixels by one raster and reads carbon at the pixels it selected, so risk, carbon and
pathway must describe the same ground cell by cell. Every read passes `like=risk`, which puts
all of them on the `prob.tif` grid. Consequence worth expecting: the Protect area measured here
is at risk-layer resolution and can differ by a fraction of a percent from the Protect area
reported by 4.1, which is measured on the pathway grid. The two are not errors of each other.

**Nodata biomass counts as zero**, the same rule 3.1 uses, with coverage measured and flagged.

### What this number leaves out

- **Peat.** On a peatland reference ecosystem the avoided emission is dominated by peat
  oxidation and fire, which can exceed the biomass pools several times over. 5.2 sees only AGB
  and BGB, so a peat Protect area is under-estimated by a wide margin. This is flagged hard, not
  footnoted, using pathway band 3.
- **Deadwood and litter**, inherited from 3.1.
- **Degradation without clearing.** The baseline is deforestation only. Forest that stays forest
  while losing carbon is a Manage question and belongs in a later component.

### Open items

0. The QB Avoided gate is a precondition only. If a future catalog has both QB Yes and QB No
   categories inside Protect, the quantification pool should be masked to the QB Yes categories
   rather than run on all Protect pixels. Moot today, since every Protect row is QB Yes.

1. The rate is extrapolated beyond the ten year window it was measured in whenever the duration
   exceeds `BASELINE_RATE_MAX_YEARS`. The tool flags this and still returns the figure. A real
   baseline would be reassessed periodically instead.
2. Timing within a year is ignored. Loss is treated as occurring at year end, and carbon as
   released in full at the moment of clearing. Belowground carbon actually decays over several
   years, which matters for a year by year credit schedule but not for a lifetime total.
3. The uniform allocation comparison is reported but not used. If the gap between ranked and
   uniform allocation turns out to be large on real sites, it is a measure of how much the
   estimate depends on the risk layer being right, and may deserve a place in the narrative.

### Example render

Single ecosystem:

> Protecting this forest ecosystem can avoid an estimated 612,000 tonnes of CO2eq emissions over
> the project's 20 year duration.

Mixed Protect area:

> Protecting this forest and peatland ecosystem can avoid an estimated 612,000 tonnes of CO2eq
> emissions over the project's 20 year duration.

No historical loss, so no baseline:

> No forest loss was recorded in this project area between 2014 and 2024, so the baseline
> projects no further loss and no avoided emissions can be claimed from protecting the standing
> forest.

| Year | Cumulative loss (ha) | Avoided this year (tCO2e) | Cumulative (tCO2e) |
|---|---|---|---|
| 1 | 14 | 26,400 | 26,400 |
| 2 | 27 | 26,100 | 52,500 |
| ... | | | |
| 19 | 246 | 34,500 | 577,800 |
| 20 | 258 | 34,200 | 612,000 |

Note the annual column rising towards the end. That is the ranking reaching denser forest, and
it is explained above.

**Narrative specified by the team**, 2026-07-21. It is deliberately one sentence and carries the
headline figure and the duration only. Everything the previous placeholder said about area,
projected loss, pools and deductions has been removed from the narrative. Most of it is still in
`values`; the deductions marker is not, by the same decision. The frontend can build any
supporting line it needs from `values`, but nothing in the component says the number is gross.

### Downstream use

The lifetime total is the headline of the Protect side of Phase 5, and feeds the Climate
Resilience and Mitigation pillar of the Triple Win framework. The annual series is chart data for
the frontend. The Manage and Restore components will produce figures on the same units and the
same duration, so the three can be presented side by side without conversion.

In [5]:
# Repeated rather than imported: notebooks cannot import each other, and the Climate notebook
# owns the same tuple for 3.1. Keep the two in step if a pool is ever added.
BIOMASS_POOLS = ("Aboveground biomass", "Belowground biomass")


@dataclass(frozen=True)
class ProjectionYear:
    """One year of the baseline projection."""

    year: int
    cumulative_loss_ha: float
    annual_avoided_tco2e: float
    cumulative_avoided_tco2e: float


def _project_loss_series(area_ha: float, rate_frac: float, years: int) -> list[float]:
    """Cumulative area lost by the end of each year, compounding.

    Takes the rate as an argument rather than reading it from a fixed source, so that swapping
    the AOI rate for a district reference table later changes the caller, not this function.
    Compounding form: the remaining stock shrinks each year, so annual loss declines. The series
    approaches `area_ha` and never exceeds it.
    """
    return [area_ha * (1.0 - math.exp(-rate_frac * t)) for t in range(1, years + 1)]


def _cumulative_carbon_by_rank(
    density_tco2e_ha: np.ndarray, risk: np.ndarray, pixel_area_ha: float
) -> np.ndarray:
    """Carbon accumulated as pixels are taken in descending risk order.

    Returns an array where element k is the total tCO2e on the k+1 highest risk pixels. Sorting
    once here is what makes the annual series cheap: every year is a lookup into this curve
    rather than a new pass over the raster.
    """
    order = np.argsort(risk)[::-1]
    return np.cumsum(density_tco2e_ha[order]) * pixel_area_ha


def _carbon_on_area(cumulative: np.ndarray, area_ha: float, pixel_area_ha: float) -> float:
    """Carbon on the highest risk `area_ha`, splitting the pixel that straddles the boundary.

    Whole pixels first, then the fraction of the next pixel needed to reach the target area.
    Splitting rather than rounding keeps the annual series smooth on small sites, where one
    pixel can be a visible share of a year of projected loss.
    """
    if area_ha <= 0 or cumulative.size == 0:
        return 0.0

    total_ha = cumulative.size * pixel_area_ha
    if area_ha >= total_ha:
        return float(cumulative[-1])

    whole = int(area_ha // pixel_area_ha)
    carbon = float(cumulative[whole - 1]) if whole > 0 else 0.0

    remainder_ha = area_ha - whole * pixel_area_ha
    if remainder_ha > 0 and whole < cumulative.size:
        prev = float(cumulative[whole - 1]) if whole > 0 else 0.0
        next_pixel_carbon = float(cumulative[whole]) - prev
        carbon += next_pixel_carbon * (remainder_ha / pixel_area_ha)

    return carbon


def _protect_qb_avoided_ha(pathway_stage: dict) -> float:
    """Area of Protect-pathway categories in the AOI that carry a QB Avoided Emissions activity.

    Reads 4.2's by_category. A category counts when its pathway is Protect and at least one of
    its catalog activities has `qb_avoided` True, i.e. the "QB Avoided Emissions" column of
    canonical_v3_activities reads Yes. Returns 0.0 when the pathway stage or 4.2 is absent, which
    makes the gate below fail closed.
    """
    try:
        by_category = component_values(pathway_stage, "4.2")["by_category"]
    except KeyError:
        return 0.0
    area_ha = 0.0
    for info in by_category.values():
        if info.get("pathway") != PATHWAY_CODES[PROTECT_CODE]:
            continue
        if any(a.get("qb_avoided") for a in info.get("activities", [])):
            area_ha += float(info.get("area_ha", 0.0))
    return area_ha


def analyze_avoided_deforestation_emissions(
    aoi: AOI, duration_years: int, rate_pct: float | None, pathway_stage: dict
) -> ComponentResult:
    """Component 5.2. Avoided emissions from unplanned deforestation on the Protect area, in tCO2e.

    `rate_pct` is the annual deforestation rate from 1.5, in percent, measured over the whole
    AOI forest. See the markdown cell for why the Protect area cannot supply its own rate.
    `pathway_stage` is the F02-P4 result, read only for the QB gate below.
    """
    component = "5.2 Avoided Emissions from Unplanned Deforestation"

    if duration_years < 1:
        raise ValueError("PROJECT_DURATION_YEARS must be a whole number of years, at least 1.")

    # Gate, at the start. Quantification runs only when the AOI holds at least one Protect
    # activity whose "QB Avoided Emissions" column is Yes. This is the flag that says avoided
    # deforestation is the right accounting method for the site. In the current canonical catalog
    # every Cat 1 (Protect) row is QB Avoided = Yes, so the gate passes whenever Protect exists
    # with a catalog row; it earns its place by failing closed if the catalog changes, or if the
    # Protect pixels have no matching activity row at all.
    qb_avoided_ha = _protect_qb_avoided_ha(pathway_stage)
    if qb_avoided_ha <= 0:
        return not_applicable(
            component,
            "No Protect-pathway activity in this project area is flagged for avoided-emissions "
            "quantification (QB Avoided Emissions = Yes), so avoided deforestation emissions are "
            "not quantified for this site.",
        )

    # The risk layer defines the working grid. It is already forest masked upstream, and it is
    # the layer the allocation ranks on, so everything else is aligned to it.
    risk_slice = load_raster_clipped(PROB_RASTER, aoi, resampling="nearest")
    pathway = load_raster_clipped(
        PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_BAND, like=risk_slice
    )

    pixel_area_ha = risk_slice.pixel_area_ha

    # Protect pixels that also carry a risk value. Protect on a non forest reference ecosystem,
    # grassland or savanna for instance, has no risk value because prob.tif is forest masked,
    # and cannot receive projected deforestation.
    protect_all = (pathway.values == PROTECT_CODE).filled(False)
    pool = protect_all & ~np.ma.getmaskarray(risk_slice.values)

    protect_all_ha = int(protect_all.sum()) * pixel_area_ha
    protect_ha = int(pool.sum()) * pixel_area_ha

    if protect_ha <= 0:
        return not_applicable(
            component,
            "No forest in this project area falls under the Protect pathway, so avoided "
            "emissions from deforestation cannot be estimated.",
        )

    if rate_pct is None:
        return not_applicable(
            component,
            "No historical deforestation rate is available for this project area, so a "
            "baseline for avoided emissions cannot be projected.",
        )

    flags: list[str] = []

    risk_coverage_pct = safe_pct(protect_ha, protect_all_ha)
    if risk_coverage_pct < PROTECT_RISK_COVERAGE_WARN_PCT:
        flags.append(
            f"5.2: the risk layer covers only {risk_coverage_pct:.0f}% of the Protect area. "
            "The remainder carries no projected loss and no avoided emissions."
        )

    if duration_years > BASELINE_RATE_MAX_YEARS:
        flags.append(
            f"5.2: the deforestation rate was measured over {DEFOR_PERIOD_YEARS} years and is "
            f"projected over {duration_years}. A rate that far outside its measurement window "
            "is an assumption, not an observation."
        )

    # Pathway band 3 does two jobs here, from one pass: it supplies the ecosystem word the
    # narrative needs, and it locates peatland.
    ecosystem = load_raster_clipped(
        PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_ECOSYSTEM_BAND, like=risk_slice
    )
    codes, counts = np.unique(ecosystem.values.filled(0)[pool].astype(int), return_counts=True)
    ecosystem_ha = {int(c): int(n) * pixel_area_ha for c, n in zip(codes, counts)}

    # Named in descending area, so the reading order matches what the site is mostly made of.
    ecosystem_words = [
        PROTECT_ECOSYSTEM_WORDS[c]
        for c in sorted(ecosystem_ha, key=ecosystem_ha.get, reverse=True)
        if c in PROTECT_ECOSYSTEM_WORDS
    ]
    ecosystem_label = oxford_join(ecosystem_words) or "natural"

    # A pool pixel outside the mapping means prob.tif and band 3 disagree about what is forest.
    unmapped_ha = sum(ha for c, ha in ecosystem_ha.items() if c not in PROTECT_ECOSYSTEM_WORDS)
    if unmapped_ha > 0:
        flags.append(
            f"5.2: {fmt_ha(unmapped_ha)} of the Protect area carries a risk value but a "
            "reference ecosystem that is not forest, mangrove or peatland. The risk layer and "
            "pathway band 3 disagree about what is forest."
        )

    peat_ha = ecosystem_ha.get(PATHWAY_ECOSYSTEM_PEATLAND, 0.0)
    if peat_ha > 0:
        flags.append(
            f"5.2: {fmt_ha(peat_ha)} of the Protect area sits on peatland. Only aboveground and "
            "belowground biomass is counted, and on peat the avoided emission is dominated by "
            "peat oxidation, so this figure is a large under-estimate there."
        )

    # Carbon density per pixel, tCO2e per hectare. Nodata is zero biomass, as in 3.1. Belowground
    # biomass is derived from aboveground by the root-to-shoot ratio (config), not read from a
    # raster, so total biomass is AGB * (1 + ratio). 5.2 only sums the two pools, so the constant
    # ratio does not distort it the way it flattens the 3.1 pool split.
    agb = load_raster_clipped(AGB_RASTER, aoi, resampling="average", like=risk_slice)
    biomass_mgha = agb.values.filled(0.0).astype(float) * (1.0 + ROOT_TO_SHOOT_RATIO)
    density = biomass_mgha * CARBON_FRACTION * CO2_PER_C

    biomass_coverage_pct = safe_pct(
        int((pool & ~np.ma.getmaskarray(agb.values)).sum()) * pixel_area_ha, protect_ha
    )
    if biomass_coverage_pct < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"5.2: the biomass raster covers only {biomass_coverage_pct:.0f}% of the Protect "
            "area. Nodata counts as zero carbon, so the estimate is an under-estimate by an "
            "unknown amount."
        )

    pool_density = density[pool]
    pool_risk = risk_slice.values.filled(0)[pool].astype(float)
    cumulative = _cumulative_carbon_by_rank(pool_density, pool_risk, pixel_area_ha)

    standing_tco2e = float(cumulative[-1])

    rate_frac = rate_pct / 100.0
    loss_series = _project_loss_series(protect_ha, rate_frac, duration_years)

    rows: list[ProjectionYear] = []
    previous_carbon = 0.0
    for year, cumulative_loss_ha in enumerate(loss_series, start=1):
        capped_ha = min(cumulative_loss_ha, protect_ha)
        carbon = _carbon_on_area(cumulative, capped_ha, pixel_area_ha)
        rows.append(
            ProjectionYear(
                year=year,
                cumulative_loss_ha=capped_ha,
                annual_avoided_tco2e=carbon - previous_carbon,
                cumulative_avoided_tco2e=carbon,
            )
        )
        previous_carbon = carbon

    total_tco2e = rows[-1].cumulative_avoided_tco2e
    projected_loss_ha = rows[-1].cumulative_loss_ha
    annual_mean_tco2e = total_tco2e / duration_years

    # Diagnostic. What the same projected loss would be worth if it were spread evenly over the
    # Protect area instead of placed on the highest risk pixels. Ranked allocation normally
    # gives the smaller number, because frontier forest carries less carbon than interior.
    uniform_tco2e = standing_tco2e * safe_pct(projected_loss_ha, protect_ha) / 100.0

    if rate_pct <= 0:
        narrative = (
            "No forest loss was recorded in this project area between 2014 and 2024, so the "
            "baseline projects no further loss and no avoided emissions can be claimed from "
            "protecting the standing forest."
        )
    else:
        narrative = (
            f"Protecting this {ecosystem_label} ecosystem can avoid an estimated "
            f"{total_tco2e:,.0f} tonnes of CO2eq emissions over the project's "
            f"{duration_years} year duration."
        )

    return ComponentResult(
        component=component,
        applicable=True,
        narrative=narrative,
        tables={"annual_projection": rows},
        values={
            "chart_series": "annual_projection",
            "chart_unit": "tCO2e",
            "chart_axis_label": "Cumulative avoided emissions (tCO2e)",
            "total_tco2e": total_tco2e,              # headline big number
            "annual_mean_tco2e": annual_mean_tco2e,
            "duration_years": duration_years,        # recorded so a saved result is reproducible
            "protect_ha": protect_ha,                # measured on the risk grid, not the 4.1 grid
            "qb_avoided_protect_ha": qb_avoided_ha,  # Protect area with a QB Avoided activity (4.2 grid)
            "protect_risk_coverage_pct": risk_coverage_pct,
            "projected_loss_ha": projected_loss_ha,
            "standing_tco2e": standing_tco2e,        # all carbon on the Protect area
            "baseline_rate_pct": rate_pct,
            "baseline_rate_source": "AOI forest 2014 to 2024, component 1.5",
            "allocation": "descending deforestation risk",
            "uniform_allocation_tco2e": uniform_tco2e,   # diagnostic, see the markdown cell
            "peat_ha": peat_ha,
            "biomass_coverage_pct": biomass_coverage_pct,
            "pools_included": list(BIOMASS_POOLS),
            "ecosystem_ha": ecosystem_ha,            # reference ecosystem split of the Protect pool
            "ecosystem_label": ecosystem_label,      # the word used in the narrative
        },
        flags=flags,
    )

In [6]:
# Run 5.2. It needs the F02-P2 general stage for the deforestation rate, and the F02-P4 pathway
# stage for the QB Avoided Emissions gate. Independent of 5.1.
try:
    results
except NameError:
    results = {}

pathway = load_results(aoi_id, STAGE_PATHWAY)
general = load_results(aoi_id, STAGE_GENERAL)
rate_pct = component_values(general, "1.5").get("rate_pct")
results["5.2"] = analyze_avoided_deforestation_emissions(
    aoi, PROJECT_DURATION_YEARS, rate_pct, pathway
)

show_result(results["5.2"])   # header, narrative, annual projection table, values, flags

[5.2 Avoided Emissions from Unplanned Deforestation]
  Protecting this forest ecosystem can avoid an estimated 5,790,414 tonnes of CO2eq emissions over the project's 20 year duration.
  annual_projection:


,year,cumulative_loss_ha,annual_avoided_tco2e,cumulative_avoided_tco2e
0,1,913.525605,274084.126500,2.740841e+05
1,2,1805.243060,286295.240572,5.603794e+05
2,3,2675.672982,285933.528914,8.463129e+05
3,4,3525.323556,284149.065477,1.130462e+06
4,5,4354.690838,284716.496203,1.415178e+06
5,6,5164.259041,284734.275784,1.699913e+06
6,7,5954.500819,285390.624269,1.985303e+06
7,8,6725.877542,284546.411757,2.269850e+06
8,9,7478.839566,283016.004135,2.552866e+06
9,10,8213.826496,283435.022509,2.836301e+06


{'chart_series': 'annual_projection',
 'chart_unit': 'tCO2e',
 'chart_axis_label': 'Cumulative avoided emissions (tCO2e)',
 'total_tco2e': 5790414.286821857,
 'annual_mean_tco2e': 289520.71434109285,
 'duration_years': 20,
 'protect_ha': 38266.8430722476,
 'qb_avoided_protect_ha': 38345.934785402074,
 'protect_risk_coverage_pct': 99.79374159582468,
 'projected_loss_ha': 14664.587626269366,
 'standing_tco2e': 20361033.173916753,
 'baseline_rate_pct': 2.4162076394938192,
 'baseline_rate_source': 'AOI forest 2014 to 2024, component 1.5',
 'allocation': 'descending deforestation risk',
 'uniform_allocation_tco2e': 7802738.119174101,
 'peat_ha': 0.0,
 'biomass_coverage_pct': 100.0,
 'pools_included': ['Aboveground biomass', 'Belowground biomass'],
 'ecosystem_ha': {1: 38266.8430722476},
 'ecosystem_label': 'forest'}

FLAG: 5.2: the deforestation rate was measured over 10 years and is projected over 20. A rate that far outside its measurement window is an assumption, not an observation.


---
## 5.3 ARR Carbon Removal (ex-ante)

Estimates the carbon a restoration project could remove over its lifetime, for the Restore areas
of the AOI where an ARR activity (active planting or Assisted Natural Regeneration) applies. This
is an ex-ante, pre-feasibility estimate, following the reference-rate method in NBS-v3-ANX-B.

**Data.** The pathway raster (band 1 pathway, band 2 ecosystem, band 3 cat_code) and the AGB
raster. No new layer, and no dependency on the 4.2 JSON. Plus the user input
`PROJECT_DURATION_YEARS`. Dryland also reads the elevation raster and the 12-band monthly
precipitation raster to derive its zone.

**Method (ANX-B Section 4), per hectare.**

```
AGB_cum  = rate_young * years_young + rate_old * years_old   # Mg d.m./ha, phases Y1-20 and Y21-40
TB_cum   = AGB_cum * (1 + R)                                 # add belowground, R = root:shoot
CO2e_cum = TB_cum * CF * 44/12                               # CF = carbon fraction
Net      = max(0, CO2e_cum - baseline_CO2e)                  # deduct biomass already on site
Net_adj  = Net * stocking_factor                            # planting 1.0, ANR/EMR 0.8
Total    = sum(Net_adj * pixel_area) over eligible pixels
low/high = Total * 0.7 / Total * 1.2
```

**Gate (ANX-B Section 3.2, not the sheet flag).** Carbon runs only for the (cat_code, ecosystem)
pairs in `ARR_SEQ_PAIRS`: Restore categories 3B, 4B, 5, 8C, 9B, 10 on Dryland, Mangrove or
Peatland, with exclusions. Peatland is TEMPORARILY excluded by team decision (2026-07-29): the
biomass method and rates exist and work, but peat is held out of quantification for now. Savanna
is deferred, its recovery is mainly soil and roots, outside the biomass scope. Cat 9B Peatland is
rewetting only with no planting in any case. So carbon is quantified for Dryland and Mangrove
only for now. This encodes the doc, NOT the `QB Carbon Sequestration` column of
canonical_v3_activities, which currently disagrees with the method on peat (sheet No, method Yes)
and savanna (sheet Yes, method defers) and is flagged for sheet reconciliation.

**Decisions locked / deferred (v1).**

- **Dryland zone derived per pixel** from elevation (metres) and the 12-band monthly
  precipitation raster, per ANX-B Section 4.5: humid montane above 1000 m; else humid lowland if
  annual rainfall is above 2000 mm and there are fewer than 3 dry months (a dry month is below
  100 mm, Walsh 1996); else seasonal lowland. Boundary and missing-data pixels fall to seasonal
  lowland, the conservative choice. Each dryland category is split into its zones, so one
  category can carry three rates across the AOI.
- **Baseline is class-based** (team decision, 2026-07-29). Net deducts a small assumed standing
  biomass per current LC state (`ARR_BASELINE_CLASS_MGHA`: C4 25, C5 5, C6 0 Mg/ha, PLACEHOLDERS
  pending references), converted `AGB * (1 + R) * CF * 44/12` and clamped at zero. This replaces
  the per-pixel AGB baseline, which on this AOI zeroed almost all vegetated Restore land because
  GEDI reads a high baseline at low biomass (Section 4.9). The per-pixel GEDI baseline and the
  gross figure are still computed and reported in `values` for comparison, switchable via
  `ARR_BASELINE_MODE`.
- **Peat temporarily excluded.** Peat carbon is held out of quantification for now by team
  decision (2026-07-29). The biomass method and rates exist; re-enable by removing peatland from
  `ARR_CARBON_DEFERRED_ECO`. Even when on, peat is biomass only: peat soil carbon and avoided
  emissions from rewetting, usually the largest peat pool, stay out of scope and are handled
  elsewhere.
- **Stocking 0.8 for ANR and mangrove EMR is uncalibrated** (doc range 0.7 to 0.85). The planting
  vs ANR split (`ARR_ANR_PAIRS`) is also uncalibrated: 3B dryland is ANR, Cat 4B/5/8C/10 mangrove
  is EMR, everything else planting.
- **Nodata AGB counts as zero baseline**, which OVER-credits (no deduction), the opposite
  direction to 5.2. Coverage is measured and flagged.
- **Beyond year 40 nothing more is credited**; the curve is defined only to Y40. A longer
  duration is flagged.
- **No leakage, uncertainty deduction, or non-permanence buffer.** The low and high (times 0.7
  and 1.2) are an indicative screening range, not a confidence interval and not a creditable
  volume.

**Example render.**

> Restoring the eligible areas of this project could remove an estimated 128,000 tCO2e over 20
> years, with an indicative range of 90,000 to 154,000 tCO2e.

**Narrative not yet specified.** Placeholder wording; replace once the team settles it.

**Downstream use.** The removal total is the Restore side of Phase 5, beside 5.2 (Protect avoided
emissions). Both are in tCO2e over the same duration, so they can be presented together.

In [7]:
# 5.3 ARR Carbon Removal (ex-ante) ------------------------------------------------------
# Reference-rate / yield-curve sequestration for Restore areas, per NBS-v3-ANX-B Section 4.
# Reads the pathway raster and AGB directly, plus elevation and monthly precipitation for the
# dryland zone. No dependency on the 4.2 JSON.


@dataclass(frozen=True)
class ArrGroup:
    """One (cat_code, ecosystem, zone) group of restoring pixels."""

    cat_code: int
    ecosystem: int
    ecosystem_label: str
    zone_label: str           # dryland zone name, or "" for mangrove and peat
    activity_mode: str        # "planting" or "ANR/EMR"
    stocking_factor: float
    area_ha: float
    net_tco2e: float          # central, after baseline and stocking
    net_tco2e_per_ha: float


@dataclass(frozen=True)
class ArrYear:
    """One year of the cumulative removal curve, summed over the AOI."""

    year: int
    cumulative_tco2e: float


def _arr_params(ecosystem: int, zone: int | None):
    """Return (rate dict, R, CF) for one ecosystem. Dryland reads by zone; others by ecosystem."""
    if ecosystem == 1:
        return (ARR_RATE_DM_DRYLAND[zone], ARR_ROOT_TO_SHOOT_DRYLAND[zone],
                ARR_CARBON_FRACTION[1])
    return ARR_RATE_DM[ecosystem], ARR_ROOT_TO_SHOOT[ecosystem], ARR_CARBON_FRACTION[ecosystem]


def _arr_accum_co2e_per_ha(rate: dict, r: float, cf: float, years: int) -> float:
    """Cumulative removal per ha before baseline and stocking, tCO2e/ha, at `years`.

    Two growth phases: Young to year 20, Old to year 40. Nothing accrues beyond year 40.
    """
    young = min(years, ARR_YOUNG_END_YEAR)
    old = max(0, min(years, ARR_OLD_END_YEAR) - ARR_YOUNG_END_YEAR)
    agb_dm = rate["young"] * young + rate["old"] * old       # Mg d.m./ha
    tb_dm = agb_dm * (1.0 + r)                                # add belowground
    return tb_dm * cf * CO2_PER_C


def _arr_baseline_co2e_per_ha(agb_mgha: np.ndarray, r: float, cf: float) -> np.ndarray:
    """Biomass carbon already on site, tCO2e/ha, from the AGB raster. Vector in, vector out."""
    return agb_mgha * (1.0 + r) * cf * CO2_PER_C


def _arr_dryland_zone(aoi: AOI, like_slice):
    """Per-pixel dryland zone code, aligned to `like_slice`, plus a validity mask.

    ANX-B Section 4.5: humid montane above 1000 m; else humid lowland if annual rainfall above
    2000 mm and fewer than 3 dry months (a dry month is below 100 mm); else seasonal lowland.
    Pixels with no elevation or precip data get the conservative default zone.
    """
    elev = load_raster_clipped(ELEVATION_RASTER, aoi, resampling="bilinear", like=like_slice)
    prec = [
        load_raster_clipped(WORLDCLIM_PREC_RASTER, aoi, resampling="average", band=b,
                            like=like_slice)
        for b in range(1, WORLDCLIM_MONTHS + 1)
    ]
    prec_stack = np.ma.stack([p.values for p in prec])           # (12, H, W)
    annual = prec_stack.sum(axis=0)                               # mm/yr
    dry_months = (prec_stack < ARR_ZONE_DRY_MONTH_MM).sum(axis=0)  # count of dry months

    inputs_valid = ~np.ma.getmaskarray(elev.values) & ~np.ma.getmaskarray(annual)
    elev_f = elev.values.filled(0.0)
    annual_f = np.ma.filled(annual, 0.0)                          # missing -> 0 -> not humid
    dry_f = np.ma.filled(dry_months, ARR_ZONE_DRY_SEASON_MONTHS)  # missing -> not humid

    zone = np.full(elev_f.shape, ARR_DRYLAND_DEFAULT_ZONE, dtype=int)
    humid = (annual_f > ARR_ZONE_WET_ANNUAL_MM) & (dry_f < ARR_ZONE_DRY_SEASON_MONTHS)
    zone[humid] = 1
    zone[elev_f > ARR_ZONE_ELEV_MONTANE_M] = 3   # elevation criterion wins over rainfall
    zone[~inputs_valid] = ARR_DRYLAND_DEFAULT_ZONE
    return zone, inputs_valid


def analyze_arr_sequestration(aoi: AOI, duration_years: int) -> ComponentResult:
    """Component 5.3. Ex-ante carbon removal from ARR restoration on the Restore area, in tCO2e."""
    component = "5.3 ARR Carbon Removal (ex-ante)"

    if duration_years < 1:
        raise ValueError("PROJECT_DURATION_YEARS must be a whole number of years, at least 1.")

    pathway = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_BAND)
    eco = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                              band=PATHWAY_ECOSYSTEM_BAND, like=pathway)
    cat = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest",
                              band=PATHWAY_CATCODE_BAND, like=pathway)
    agb = load_raster_clipped(AGB_RASTER, aoi, resampling="average", like=pathway)
    pix = pathway.pixel_area_ha

    restore = (pathway.values == RESTORE_CODE).filled(False)
    if not restore.any():
        return not_applicable(
            component,
            "No area of this project falls under the Restore pathway, so ARR carbon removal "
            "cannot be estimated.",
        )

    catv = cat.values.filled(0).astype(int)
    ecov = eco.values.filled(0).astype(int)
    agbv = agb.values.filled(0.0).astype(float)
    agb_valid = ~np.ma.getmaskarray(agb.values)

    # Dryland zone, derived once if any dryland Restore pixel exists.
    dryland_restore = restore & (ecov == 1)
    if dryland_restore.any():
        zone_arr, zone_valid = _arr_dryland_zone(aoi, pathway)
    else:
        zone_arr = np.full(restore.shape, ARR_DRYLAND_DEFAULT_ZONE, dtype=int)
        zone_valid = np.ones(restore.shape, dtype=bool)

    groups: list[ArrGroup] = []
    quantified = np.zeros_like(restore, dtype=bool)
    annual = np.zeros(duration_years, dtype=float)
    agb_valid_quant_ha = 0.0
    gross_tco2e = 0.0        # before baseline deduction, for diagnosis
    clamped_ha = 0.0         # area where baseline >= accumulation, net forced to zero
    net_classbaseline_tco2e = 0.0   # diagnostic: net under a small class-based baseline
    net_gedi_tco2e = 0.0            # diagnostic: net under the per-pixel GEDI baseline

    for cc, ec in sorted(ARR_SEQ_PAIRS):
        if ec in ARR_CARBON_DEFERRED_ECO:
            continue  # ecosystem held out of quantification (savanna deferred, peat excluded)
        base_mask = restore & (catv == cc) & (ecov == ec)
        if not base_mask.any():
            continue
        # Dryland splits into its three zones; other ecosystems are a single group.
        subgroups = ([(z, base_mask & (zone_arr == z)) for z in ARR_DRYLAND_ZONES]
                     if ec == 1 else [(None, base_mask)])

        for zone_code, gmask in subgroups:
            n = int(gmask.sum())
            if n == 0:
                continue
            quantified |= gmask
            area = n * pix
            rate, r, cf = _arr_params(ec, zone_code)
            conv = (1.0 + r) * cf * CO2_PER_C
            stocking = ARR_STOCKING_ANR if (cc, ec) in ARR_ANR_PAIRS else ARR_STOCKING_PLANTING
            mode = "ANR/EMR" if (cc, ec) in ARR_ANR_PAIRS else "planting"
            accum = _arr_accum_co2e_per_ha(rate, r, cf, duration_years)

            # Three baselines. The primary one is chosen by ARR_BASELINE_MODE; the other two
            # travel in values for comparison. base_class and base_none are constant per pixel.
            cstate = ARR_RESTORE_CAT_CSTATE.get(cc)
            base_gedi = agbv[gmask] * conv                                     # per-pixel vector
            base_class = np.full(n, ARR_BASELINE_CLASS_MGHA.get(cstate, 0.0) * conv)
            base_primary = {"class": base_class, "per_pixel_agb": base_gedi,
                            "none": np.zeros(n)}[ARR_BASELINE_MODE]

            def _net(b, a=accum, st=stocking):
                return float(np.maximum(0.0, a - b).sum()) * st * pix

            net = _net(base_primary)
            gross_tco2e += accum * stocking * area                            # baseline = 0
            net_gedi_tco2e += _net(base_gedi)
            net_classbaseline_tco2e += _net(base_class)
            clamped_ha += int((base_gedi >= accum).sum()) * pix               # GEDI diagnostic
            groups.append(ArrGroup(
                cat_code=cc, ecosystem=ec,
                ecosystem_label=PATHWAY_ECOSYSTEM_CODES.get(ec, f"eco {ec}"),
                zone_label=(ARR_DRYLAND_ZONES[zone_code] if ec == 1 else ""),
                activity_mode=mode, stocking_factor=stocking,
                area_ha=area, net_tco2e=net,
                net_tco2e_per_ha=(net / area if area else 0.0),
            ))

            for i, t in enumerate(range(1, duration_years + 1)):
                accum_t = _arr_accum_co2e_per_ha(rate, r, cf, t)
                annual[i] += float(np.maximum(0.0, accum_t - base_primary).sum()) * stocking * pix

            agb_valid_quant_ha += int((gmask & agb_valid).sum()) * pix

    if not groups:
        return not_applicable(
            component,
            "No Restore area carries an ARR activity eligible for carbon sequestration (planting "
            "or ANR on dryland or mangrove), so ex-ante carbon removal cannot be estimated.",
        )

    total = sum(g.net_tco2e for g in groups)
    quantified_ha = sum(g.area_ha for g in groups)

    # Restore area eligible for an activity but not carbon quantified.
    deferred = restore & ~quantified
    savanna_ha = int((deferred & (ecov == 4)).sum()) * pix
    peat_ha = int((deferred & (ecov == PATHWAY_ECOSYSTEM_PEATLAND)).sum()) * pix
    other_deferred_ha = int(deferred.sum()) * pix - savanna_ha - peat_ha

    # Dryland pixels that could not be zoned and fell to the default.
    zone_defaulted_ha = int((quantified & (ecov == 1) & ~zone_valid).sum()) * pix

    flags: list[str] = []
    cov = safe_pct(agb_valid_quant_ha, quantified_ha)
    if ARR_BASELINE_MODE == "per_pixel_agb" and cov < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"5.3: the AGB raster covers only {cov:.0f}% of the quantified area. Missing AGB is "
            "read as zero baseline, so no standing biomass is deducted there and the removal is "
            "over-estimated by an unknown amount."
        )
    flags.append(
        f"5.3: baseline mode is '{ARR_BASELINE_MODE}'. The class-based values are PLACEHOLDERS "
        f"(C4 {ARR_BASELINE_CLASS_MGHA['C4']:g}, C5 {ARR_BASELINE_CLASS_MGHA['C5']:g}, "
        f"C6 {ARR_BASELINE_CLASS_MGHA['C6']:g} Mg/ha) pending references, so the total is "
        f"indicative. For comparison, the per-pixel GEDI baseline gives {net_gedi_tco2e:,.0f} "
        f"tCO2e and gross (no baseline) {gross_tco2e:,.0f}."
    )
    if savanna_ha > 0:
        flags.append(
            f"5.3: {fmt_ha(savanna_ha)} of Restore area is savanna, whose carbon is deferred "
            "(recovery is mainly soil and roots). Activity and benefits still apply."
        )
    if peat_ha > 0:
        flags.append(
            f"5.3: {fmt_ha(peat_ha)} of Restore area is peatland, temporarily excluded from "
            "carbon quantification by team decision (the biomass method and rates exist; remove "
            "peatland from ARR_CARBON_DEFERRED_ECO to re-enable). Activity and benefits still apply."
        )
    if zone_defaulted_ha > 0:
        flags.append(
            f"5.3: {fmt_ha(zone_defaulted_ha)} of dryland could not be zoned (missing elevation "
            "or precipitation) and defaulted to the seasonal-lowland rate."
        )
    if duration_years > ARR_OLD_END_YEAR:
        flags.append(
            f"5.3: the accumulation curve is defined only to year {ARR_OLD_END_YEAR}; the "
            f"{duration_years - ARR_OLD_END_YEAR} years beyond it add no further removal."
        )
    flags.append(
        "5.3: the ANR/EMR stocking factor (0.8) and the planting-vs-ANR split are uncalibrated. "
        "Dryland zones use elevation and 12-band monthly precipitation (units to verify), with a "
        "dry month defined as below 100 mm (Walsh 1996)."
    )

    total_low = total * ARR_UNCERTAINTY_LOW
    total_high = total * ARR_UNCERTAINTY_HIGH

    net_by_ecosystem: dict[str, float] = {}
    area_by_ecosystem: dict[str, float] = {}
    area_by_dryland_zone: dict[str, float] = {}
    for g in groups:
        net_by_ecosystem[g.ecosystem_label] = net_by_ecosystem.get(g.ecosystem_label, 0.0) + g.net_tco2e
        area_by_ecosystem[g.ecosystem_label] = area_by_ecosystem.get(g.ecosystem_label, 0.0) + g.area_ha
        if g.zone_label:
            area_by_dryland_zone[g.zone_label] = area_by_dryland_zone.get(g.zone_label, 0.0) + g.area_ha

    # Placeholder wording, see the markdown cell.
    narrative = (
        f"Restoring the eligible areas of this project could remove an estimated {total:,.0f} "
        f"tCO2e over {duration_years} years, with an indicative range of {total_low:,.0f} to "
        f"{total_high:,.0f} tCO2e."
    )

    groups_sorted = sorted(groups, key=lambda g: -g.net_tco2e)
    return ComponentResult(
        component=component,
        applicable=True,
        narrative=narrative,
        tables={
            "annual_projection": [ArrYear(t + 1, annual[t]) for t in range(duration_years)],
            "groups": groups_sorted,
        },
        values={
            "total_tco2e": total,                # headline central estimate
            "total_low_tco2e": total_low,
            "total_high_tco2e": total_high,
            "duration_years": duration_years,
            "quantified_ha": quantified_ha,
            "baseline_mode": ARR_BASELINE_MODE,
            "total_gross_tco2e": gross_tco2e,
            "net_perpixel_gedi_tco2e": net_gedi_tco2e,
            "net_classbaseline_tco2e": net_classbaseline_tco2e,
            "gedi_baseline_zeroed_ha": clamped_ha,   # diagnostic on the GEDI baseline
            "net_by_ecosystem_tco2e": net_by_ecosystem,
            "area_by_ecosystem_ha": area_by_ecosystem,
            "area_by_dryland_zone_ha": area_by_dryland_zone,
            "agb_coverage_pct": cov,
            "zone_defaulted_ha": zone_defaulted_ha,
            "deferred_savanna_ha": savanna_ha,
            "deferred_peat_ha": peat_ha,
            "peat_excluded": True,
            "deferred_other_ha": other_deferred_ha,
            "method": "reference-rate / yield-curve, NBS-v3-ANX-B v2",
            "pools_included": ["aboveground biomass", "belowground biomass"],
            "pools_excluded": ["soil organic carbon", "peat soil", "dead wood", "litter",
                               "avoided emissions"],
            "uncertainty": {"low": ARR_UNCERTAINTY_LOW, "high": ARR_UNCERTAINTY_HIGH},
        },
        flags=flags,
    )

In [8]:
# Run 5.3. It reads the pathway raster and AGB directly, so it needs no earlier stage file,
# only the AOI and PROJECT_DURATION_YEARS. Independent of 5.1 and 5.2.
try:
    results
except NameError:
    results = {}

results["5.3"] = analyze_arr_sequestration(aoi, PROJECT_DURATION_YEARS)

r = results["5.3"]
v = r.values
print(f"[5.3] {r.component}{'' if r.applicable else '  (not applicable)'}")
print("     ", r.narrative)
if r.applicable:
    gross = v["total_gross_tco2e"]
    removed = 100 * (1 - v["total_tco2e"] / gross) if gross else 0.0
    print(f"      quantified area : {v['quantified_ha']:,.0f} ha   AGB coverage {v['agb_coverage_pct']:.0f}%")
    print(f"      net vs gross    : {v['total_tco2e']:,.0f}  vs  {gross:,.0f} tCO2e  (baseline removed {removed:.0f}%)")
    print(f"      baseline mode   : {v['baseline_mode']} (primary)")
    print(f"      --- baseline comparison (net tCO2e over {v['duration_years']} yr) ---")
    print(f"        class-based baseline (primary) : {v['net_classbaseline_tco2e']:,.0f}")
    print(f"        per-pixel GEDI baseline        : {v['net_perpixel_gedi_tco2e']:,.0f}")
    print(f"        gross, no baseline             : {v['total_gross_tco2e']:,.0f}")
    for lbl, a in v["area_by_dryland_zone_ha"].items():
        print(f"        dryland {lbl}: {a:,.0f} ha")
    print(f"      deferred        : savanna {v['deferred_savanna_ha']:,.0f} ha, "
          f"peat {v['deferred_peat_ha']:,.0f} ha, other {v['deferred_other_ha']:,.0f} ha")
for f in r.flags:
    print("FLAG:", f)

[5.3] 5.3 ARR Carbon Removal (ex-ante)
      Restoring the eligible areas of this project could remove an estimated 337 tCO2e over 20 years, with an indicative range of 236 to 404 tCO2e.
      quantified area : 16,446 ha   AGB coverage 100%
      net vs gross    : 337  vs  2,141,162 tCO2e  (baseline removed 100%)
      zero-net area   : 16,384 ha (baseline >= 20yr accumulation)
      --- baseline comparison (net tCO2e over 20 yr) ---
        per-pixel GEDI baseline (current): 337
        small class-based baseline       : 1,470,591
        gross, no baseline               : 2,141,162
        dryland humid lowland: 16,446 ha
      deferred        : savanna 0 ha, peat 0 ha, other 0 ha
FLAG: 5.3: on 16,384 ha of the quantified area the standing biomass (AGB baseline) already meets or exceeds the modelled 20-year accumulation, so its net removal is zero. Net total is 337 tCO2e against a gross of 2,141,162 before the baseline deduction.
FLAG: 5.3: the ANR/EMR stocking factor (0.8) and the p

---
## Save

Writes whatever is in `results` to `outputs/<aoi_id>__F02-P5-benefit.json`. Run 5.1 and 5.2
above first; running only one of them saves only that one.

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_BENEFIT)
print(f"Saved {path} with: {', '.join(results)}")